# ERS-CPS Notebook: Energy-Based Safety and Resilience Analysis

This notebook provides a fully reproducible implementation of a risk-aware, energy-based early-warning framework for cyber–physical systems.

## What this notebook does
- Models system energy and instability
- Computes Critical Transition Indicator (CTI)
- Performs nonlinear stability analysis
- Evaluates early warning capability
- Generates figures and tables for analysis

## How to use
Run all cells sequentially. Outputs (figures and tables) will be saved automatically.


# A Risk-Aware Visual Analytics Framework for Safety and Resilience Assessment in Critical Cyber-Physical Infrastructure Systems

## v17 Notebook: Stability-Aware Physics-Informed Dynamics with Google Drive Outputs

This v17 notebook extends the previous prototype by introducing a stronger analytical core for **instability growth, resilience dissipation, and critical transition detection**. The notebook is organized around four coupled layers:

1. **Cascade-Aware Safety Risk Score (CASRS)** for local-to-system risk aggregation.  
2. **Energy-based resilience dynamics** for temporal degradation, dissipation, and memory.  
3. **Stability-aware analytical law** for next-step energy growth and instability margin.  
4. **Physics-informed hybrid intelligence** for residual correction and early-warning ranking.

All outputs are saved automatically to **Google Drive** when the notebook is executed in Colab, including figures, CSV tables, and a compact text report.


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:
# ============================================================
# 1. ENVIRONMENT SETUP
# ============================================================

import os
import json
import math
import time
import random
import hashlib
import warnings
from dataclasses import dataclass
from pathlib import Path
from collections import defaultdict, deque

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

IN_COLAB = False
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

def setup_output_dirs(base_output: Path):
    fig_dir = base_output / "figures"
    tab_dir = base_output / "tables"
    oth_dir = base_output / "others"
    for d in [base_output, fig_dir, tab_dir, oth_dir]:
        d.mkdir(parents=True, exist_ok=True)
    return fig_dir, tab_dir, oth_dir

BASE_OUTPUT = None
FIG_DIR = None
TAB_DIR = None
OTH_DIR = None

def save_csv(df, filename, index=False):
    path = TAB_DIR / filename
    df.to_csv(path, index=index)
    return path

def save_text_report(filename, text):
    path = OTH_DIR / filename
    path.write_text(text, encoding="utf-8")
    return path

def save_figure(fig, filename, dpi=300):
    path = FIG_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi, bbox_inches="tight")
    return path

print("=" * 100)
print("Environment imports ready")
print(f"IN_COLAB: {IN_COLAB}")
print("=" * 100)


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:
# ============================================================
# 2. GOOGLE DRIVE MOUNT + OUTPUT DIRECTORY INITIALIZATION
# ============================================================

if IN_COLAB:
    try:
        from google.colab import drive  # type: ignore

        # If already mounted, this may raise an exception. That is fine as long
        # as /content/drive/MyDrive is available afterward.
        try:
            drive.mount('/content/drive', force_remount=False)
            print("Google Drive mount attempted.")
        except Exception as e:
            print(f"[WARN] drive.mount returned: {e}")

        if os.path.exists("/content/drive/MyDrive"):
            BASE_OUTPUT = Path("/content/drive/MyDrive/Outputs/JSSR_STIN_Safety_Notebook_v17")
            print("Using Google Drive output folder.")
        else:
            BASE_OUTPUT = Path("./JSSR_STIN_Safety_Notebook_v17_local_fallback")
            print("[WARN] Google Drive path unavailable. Falling back to local output folder.")
    except Exception as e:
        BASE_OUTPUT = Path("./JSSR_STIN_Safety_Notebook_v17_local_fallback")
        print(f"[WARN] Google Drive initialization failed: {e}")
        print("Falling back to local output folder.")
else:
    BASE_OUTPUT = Path("./JSSR_STIN_Safety_Notebook_v17")
    print("Not running in Colab; local directories will be used.")

FIG_DIR, TAB_DIR, OTH_DIR = setup_output_dirs(BASE_OUTPUT)

print("=" * 100)
print(f"BASE_OUTPUT: {BASE_OUTPUT.resolve()}")
print(f"FIG_DIR: {FIG_DIR.resolve()}")
print(f"TAB_DIR: {TAB_DIR.resolve()}")
print(f"OTH_DIR: {OTH_DIR.resolve()}")
print("=" * 100)


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# 3. CONFIGURATION
# ============================================================

CONFIG = {
    "n_records": 450,
    "n_nodes": 18,
    "time_step_seconds": 60,
    "rolling_window": 20,
    "trust_decay_minor": 0.025,
    "trust_decay_major": 0.060,
    "trust_recovery_rate": 0.012,
    "latency_base_ms": 85.0,
    "latency_noise_ms": 9.0,
    "packet_loss_base": 0.004,
    "max_hops": 3,
    "scenario_lengths": {
        "nominal": 140,
        "degraded": 150,
        "adversarial": 160,
    },
    "risk_state_thresholds": {
        "stable": 0.25,
        "stressed": 0.50,
        "critical": 0.75,
    },
    "alert_budget_per_step": 4,
    "critical_failure_threshold": 0.74,
    "critical_memory_threshold": 0.60,
    "acceleration_weight": 0.15,
    "adaptive_stress_weight": 0.50,
    "system_pressure_weight": 0.20,
    "failure_accumulator_threshold": 0.60,
}

print(json.dumps(CONFIG, indent=2))
print(f"Total scenario steps = {sum(CONFIG['scenario_lengths'].values())}")


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# 4. DATA ACQUISITION: NASA CMR WITH SAFE FALLBACK
# ============================================================

def fetch_nasa_cmr_metadata(limit=450):
    """
    Colab-friendly NASA CMR retrieval.
    Falls back to synthetic metadata if retrieval fails.
    """
    url = "https://cmr.earthdata.nasa.gov/search/granules.umm_json"
    params = {
        "page_size": min(limit, 200),
        "provider": "NASA",
    }

    try:
        import requests  # local import for robustness
        resp = requests.get(url, params=params, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        items = payload.get("items", [])
        print(f"[INFO] NASA CMR entries fetched: {len(items)}")
        if len(items) == 0:
            raise ValueError("CMR returned zero entries.")

        entries = []
        for item in items:
            meta = item.get("umm", {})
            provider_meta = meta.get("ProviderDates", [])
            temporal = meta.get("TemporalExtent", {})
            range_dt = temporal.get("RangeDateTime", {})
            related_urls = meta.get("RelatedUrls", [])
            platforms = meta.get("Platforms", [])

            platform_name = "NA"
            if platforms:
                platform_name = platforms[0].get("ShortName", "NA")

            entries.append({
                "id": item.get("meta", {}).get("concept-id", "NA"),
                "producer_granule_id": meta.get("GranuleUR", "NA"),
                "time_start": range_dt.get("BeginningDateTime", "NA"),
                "updated": provider_meta[0].get("Date", "NA") if provider_meta else "NA",
                "data_center": meta.get("DataGranule", {}).get("DayNightFlag", "NA"),
                "processing_level_id": meta.get("CollectionReference", {}).get("Version", "NA"),
                "archive_center": related_urls[0].get("URLContentType", "NA") if related_urls else "NA",
                "platform": platform_name,
                "short_name": meta.get("CollectionReference", {}).get("ShortName", "NA"),
                "version_id": meta.get("CollectionReference", {}).get("Version", "NA"),
            })

        if len(entries) < limit:
            print(f"[INFO] Using {len(entries)} CMR rows; fewer than requested.")
        return entries[:limit]
    except Exception as e:
        print(f"[WARN] NASA CMR retrieval failed. Using synthetic metadata. Reason: {e}")
        return None


def build_synthetic_metadata(limit=450):
    providers = ["NASA_GES_DISC", "NASA_LARC", "NASA_LPDAAC", "NASA_OB_DAAC", "NASA_CDDIS"]
    levels = ["L1", "L2", "L3", "L4"]
    archives = ["ARC_A", "ARC_B", "ARC_C", "ARC_D"]
    platforms = ["LEO_A", "LEO_B", "LEO_C", "UAV_RELAY", "GROUND_GATEWAY"]
    products = ["AEROSOL", "ATMOS", "TEMP", "SOIL", "OCEAN", "CLOUD", "ICE"]

    rows = []
    base_ts = pd.Timestamp("2025-01-01")
    for i in range(limit):
        rows.append({
            "id": f"SYN_{i:05d}",
            "producer_granule_id": f"GRANULE_{100000+i}",
            "time_start": (base_ts + pd.Timedelta(minutes=15 * i)).isoformat(),
            "updated": (base_ts + pd.Timedelta(minutes=15 * i + 2)).isoformat(),
            "data_center": random.choice(providers),
            "processing_level_id": random.choice(levels),
            "archive_center": random.choice(archives),
            "platform": random.choice(platforms),
            "short_name": random.choice(products),
            "version_id": str(random.randint(1, 9)),
        })
    print(f"[INFO] Synthetic metadata generated: {len(rows)}")
    return rows


raw_entries = fetch_nasa_cmr_metadata(limit=CONFIG["n_records"])
if raw_entries is None or len(raw_entries) < CONFIG["n_records"]:
    if raw_entries is None:
        raw_entries = []
    needed = CONFIG["n_records"] - len(raw_entries)
    if needed > 0:
        synth = build_synthetic_metadata(limit=needed)
        raw_entries = list(raw_entries) + synth

print(f"Sample raw entries available: {len(raw_entries)}")
print("First raw entry keys:")
print(list(raw_entries[0].keys())[:20])


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# 5. NORMALIZATION
# ============================================================

def normalize_entry(entry):
    def _get(d, *keys, default="NA"):
        for k in keys:
            if k in d and d[k] not in [None, ""]:
                return d[k]
        return default

    normalized = {
        "record_id": str(_get(entry, "id", "producer_granule_id", default="NA")),
        "product_id": str(_get(entry, "producer_granule_id", "short_name", default="NA")),
        "acquisition_time": str(_get(entry, "time_start", "updated", default="NA")),
        "archive_center": str(_get(entry, "archive_center", "data_center", default="NA")),
        "processing_level": str(_get(entry, "processing_level_id", default="NA")),
        "platform": str(_get(entry, "platform", default="NA")),
        "version": str(_get(entry, "version_id", default="NA")),
        "collection": str(_get(entry, "short_name", default="NA")),
    }
    return normalized


norm_rows = [normalize_entry(e) for e in raw_entries]
meta_df = pd.DataFrame(norm_rows)
meta_df["acquisition_time"] = pd.to_datetime(meta_df["acquisition_time"], errors="coerce")
meta_df["acquisition_time"] = meta_df["acquisition_time"].fillna(pd.Timestamp("2025-01-01"))
meta_df = meta_df.sort_values("acquisition_time").reset_index(drop=True)

print("=" * 100)
print("Normalized metadata shape:", meta_df.shape)
print(meta_df.head(5))
print("=" * 100)
print("Null counts:")
print(meta_df.isna().sum())
print("=" * 100)

meta_df.to_csv(TAB_DIR / "normalized_metadata_preview.csv", index=False)
print(f"Saved normalized metadata preview to: {TAB_DIR / 'normalized_metadata_preview.csv'}")


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# 6. CRITICAL INFRASTRUCTURE GRAPH CONSTRUCTION
# ============================================================

def build_infrastructure_graph(n_nodes=18, seed=42):
    rng = np.random.default_rng(seed)
    G = nx.barabasi_albert_graph(n=n_nodes, m=2, seed=seed)
    G = nx.relabel_nodes(G, {n: f"N{n:02d}" for n in G.nodes()})

    node_types = (
        ["SAT"] * max(4, n_nodes // 4) +
        ["AERIAL"] * max(3, n_nodes // 5) +
        ["GROUND"] * (n_nodes - max(4, n_nodes // 4) - max(3, n_nodes // 5))
    )
    rng.shuffle(node_types)

    for i, node in enumerate(G.nodes()):
        node_type = node_types[i]
        criticality = float(np.clip(rng.normal(0.6 if node_type == "GROUND" else 0.7, 0.12), 0.2, 1.0))
        redundancy = float(np.clip(rng.normal(0.55, 0.18), 0.05, 1.0))
        exposure = float(np.clip(rng.normal(0.45 if node_type == "GROUND" else 0.60, 0.15), 0.05, 1.0))
        G.nodes[node]["node_type"] = node_type
        G.nodes[node]["criticality"] = criticality
        G.nodes[node]["redundancy"] = redundancy
        G.nodes[node]["exposure"] = exposure

    between = nx.betweenness_centrality(G, normalized=True)
    degree = dict(G.degree())
    max_deg = max(degree.values())

    for node in G.nodes():
        G.nodes[node]["betweenness"] = float(between[node])
        G.nodes[node]["degree_norm"] = float(degree[node] / max_deg)

    return G


G = build_infrastructure_graph(n_nodes=CONFIG["n_nodes"], seed=SEED)

node_info_df = pd.DataFrame([
    {
        "node": n,
        **G.nodes[n],
    }
    for n in G.nodes()
]).sort_values(["criticality", "betweenness"], ascending=[False, False]).reset_index(drop=True)

print("=" * 100)
print("Infrastructure graph summary")
print(f"Nodes: {G.number_of_nodes()} | Edges: {G.number_of_edges()}")
print(node_info_df.head(10))
print("=" * 100)

node_info_df.to_csv(TAB_DIR / "node_attributes.csv", index=False)
print(f"Saved node attributes to: {TAB_DIR / 'node_attributes.csv'}")


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# 7. NOVEL IDEA: CASCADE-AWARE SAFETY RISK AND EARLY WARNING
# ============================================================

"""
Novel notebook concept:
Instead of only tracking trust/integrity, this notebook introduces a
Cascade-Aware Safety Risk Score (CASRS) for critical infrastructure STIN analysis.

CASRS combines:
1) local integrity violations,
2) trust degradation,
3) performance degradation,
4) network criticality/exposure,
5) graph-based failure propagation pressure,
6) resilience memory (persistence across time),
7) system-level stress field (global degradation pressure).

This directly follows the journal-oriented changes:
- failure precursors,
- cascading disruption,
- resilience degradation,
- early warning,
- human-centric prioritized alerts.
"""

def stable_hash(payload_dict):
    payload_str = json.dumps(payload_dict, sort_keys=True)
    return hashlib.sha256(payload_str.encode("utf-8")).hexdigest()


def scenario_label(step):
    n1 = CONFIG["scenario_lengths"]["nominal"]
    n2 = n1 + CONFIG["scenario_lengths"]["degraded"]
    if step < n1:
        return "nominal"
    elif step < n2:
        return "degraded"
    return "adversarial"


def get_attack_profile(step, node_name):
    scen = scenario_label(step)
    node = G.nodes[node_name]
    criticality = node["criticality"]
    exposure = node["exposure"]
    degree_norm = node["degree_norm"]

    if scen == "nominal":
        attack_prob = 0.015 + 0.015 * exposure
        perf_drift = 0.01
        coordination_factor = 0.00
    elif scen == "degraded":
        attack_prob = 0.055 + 0.055 * exposure + 0.02 * degree_norm + 0.05
        perf_drift = 0.11
        coordination_factor = 0.10
    else:
        attack_prob = 0.10 + 0.10 * exposure + 0.05 * degree_norm + 0.05 * criticality + 0.15
        perf_drift = 0.24
        coordination_factor = 0.25

    return {
        "scenario": scen,
        "attack_prob": float(np.clip(attack_prob, 0.0, 0.85)),
        "perf_drift": perf_drift,
        "coordination_factor": coordination_factor,
    }


def classify_risk_state(x, thresholds):
    if x >= thresholds["critical"]:
        return "critical"
    elif x >= thresholds["stressed"]:
        return "stressed"
    elif x >= 0.40:
        return "pre-critical"
    elif x >= thresholds["stable"]:
        return "degraded"
    return "stable"


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# 8. SIMULATION ENGINE
# ============================================================

def simulate_safety_resilience(meta_df, G, config):
    n_steps = sum(config["scenario_lengths"].values())
    assert n_steps <= len(meta_df), "Not enough metadata rows for configured scenario lengths."

    node_states = {
        node: {
            "trust": 1.0,
            "risk_memory": 0.0,
            "stress_memory": 0.0,
            "last_latency": config["latency_base_ms"],
            "last_packet_loss": config["packet_loss_base"],
            "recovery_counter": 0,
        }
        for node in G.nodes()
    }

    results = []
    failures_triggered = []
    alert_log = []

    ordered_nodes = list(G.nodes())
    metadata_slice = meta_df.iloc[:n_steps].copy().reset_index(drop=True)

    for step in range(n_steps):
        row = metadata_slice.iloc[step].to_dict()
        current_time = row["acquisition_time"] + pd.Timedelta(seconds=config["time_step_seconds"] * step)
        scen = scenario_label(step)

        system_stress = float(np.mean([1.0 - node_states[n]["trust"] for n in G.nodes()]))
        system_pressure = config["system_pressure_weight"] * system_stress

        targeted_nodes = set()
        if scen == "adversarial":
            high_priority = node_info_df.sort_values(
                ["criticality", "betweenness", "exposure"], ascending=False
            )["node"].tolist()[: max(3, len(ordered_nodes) // 4)]
            targeted_nodes = set(high_priority)

        step_rows = []
        for idx, node in enumerate(ordered_nodes):
            state = node_states[node]
            node_attr = G.nodes[node]
            profile = get_attack_profile(step, node)

            payload = {
                "record_id": row["record_id"],
                "product_id": row["product_id"],
                "archive_center": row["archive_center"],
                "processing_level": row["processing_level"],
                "platform": row["platform"],
                "node": node,
                "step": step,
                "scenario": scen,
            }

            clean_hash = stable_hash(payload)

            attack_prob = profile["attack_prob"]
            if node in targeted_nodes:
                attack_prob += profile["coordination_factor"]
            attack_prob = float(np.clip(attack_prob, 0.0, 0.90))

            integrity_ok = np.random.rand() >= attack_prob

            tamper_severity = 0.0
            if not integrity_ok:
                tamper_severity = float(np.clip(
                    0.45 + 0.25 * node_attr["criticality"] + 0.20 * np.random.rand(),
                    0.0, 1.0
                ))
                tampered_payload = payload.copy()
                tampered_payload["archive_center"] = f"{payload['archive_center']}_TAMPERED"
                observed_hash = stable_hash(tampered_payload)
            else:
                observed_hash = clean_hash

            if integrity_ok:
                trust = min(1.0, state["trust"] + config["trust_recovery_rate"] * (1.0 - state["trust"]))
                state["recovery_counter"] += 1
            else:
                decay = config["trust_decay_minor"] + tamper_severity * (config["trust_decay_major"] - config["trust_decay_minor"])
                trust = max(0.0, state["trust"] - decay)
                state["recovery_counter"] = 0

            latency = (
                config["latency_base_ms"]
                + np.random.normal(0.0, config["latency_noise_ms"])
                + 65.0 * (1.0 - trust)
                + 45.0 * profile["perf_drift"]
                + 18.0 * node_attr["degree_norm"]
                + 22.0 * system_stress
            )
            packet_loss = (
                config["packet_loss_base"]
                + 0.08 * (1.0 - trust)
                + 0.045 * profile["perf_drift"]
                + 0.020 * system_stress
                + np.abs(np.random.normal(0, 0.004))
            )
            latency = float(max(1.0, latency))
            packet_loss = float(np.clip(packet_loss, 0.0, 1.0))

            neigh = list(G.neighbors(node))
            if len(neigh) > 0:
                neighbor_terms = []
                for n in neigh:
                    n_attr = G.nodes[n]
                    n_state = node_states[n]
                    influence = (
                        (1.0 - n_state["trust"]) *
                        (0.6 * n_attr["criticality"] + 0.4 * n_attr["betweenness"])
                    )
                    neighbor_terms.append(influence)
                neighbor_pressure = float(np.sum(neighbor_terms) / (len(neighbor_terms) + 1e-6))
            else:
                neighbor_pressure = 0.0

            local_disruption = (
                0.30 * (0 if integrity_ok else 1)
                + 0.20 * tamper_severity
                + 0.20 * (1.0 - trust)
                + 0.15 * min(1.0, latency / 250.0)
                + 0.10 * min(1.0, packet_loss / 0.20)
            )

            propagation_pressure = float(np.clip(neighbor_pressure, 0.0, 1.5))
            exposure_factor = 0.15 * node_attr["exposure"]
            criticality_factor = 0.15 * node_attr["criticality"]

            casrs = (
                0.38 * local_disruption
                + 0.20 * min(1.0, propagation_pressure)
                + exposure_factor
                + criticality_factor
                + 0.10 * state["risk_memory"]
                + system_pressure
            )

            risk_delta = casrs - state["risk_memory"]
            acceleration = max(0.0, risk_delta)
            casrs = casrs + config["acceleration_weight"] * acceleration

            adaptive_factor = 1.0 + config["adaptive_stress_weight"] * state["stress_memory"]
            casrs = casrs * adaptive_factor
            casrs = float(np.clip(casrs, 0.0, 1.0))

            # Counterfactual perturbation sensitivity
            cf_trust = max(0.0, trust - 0.08)
            cf_latency = latency * 1.10
            cf_packet_loss = min(1.0, packet_loss * 1.15)

            cf_local_disruption = (
                0.30 * (0 if integrity_ok else 1)
                + 0.20 * tamper_severity
                + 0.20 * (1.0 - cf_trust)
                + 0.15 * min(1.0, cf_latency / 250.0)
                + 0.10 * min(1.0, cf_packet_loss / 0.20)
            )

            cf_casrs = (
                0.38 * cf_local_disruption
                + 0.20 * min(1.0, propagation_pressure)
                + exposure_factor
                + criticality_factor
                + 0.10 * state["risk_memory"]
                + system_pressure
            )
            cf_casrs = float(np.clip(cf_casrs, 0.0, 1.0))
            counterfactual_sensitivity = float(max(0.0, cf_casrs - casrs))

            baseline_risk = (
                0.60 * (1.0 - trust)
                + 0.20 * (0 if integrity_ok else 1)
            )
            baseline_risk = float(np.clip(baseline_risk, 0.0, 1.0))

            resilience_score = float(np.clip(1.0 - casrs, 0.0, 1.0))
            risk_state = classify_risk_state(casrs, config["risk_state_thresholds"])
            baseline_state = classify_risk_state(baseline_risk, config["risk_state_thresholds"])

            failure_accumulator = 0.60 * state["risk_memory"] + 0.40 * state["stress_memory"]
            critical_failure = int(
                casrs >= config["critical_failure_threshold"] and
                failure_accumulator >= config["failure_accumulator_threshold"]
            )

            if critical_failure == 1:
                failures_triggered.append({
                    "step": step,
                    "time": current_time,
                    "node": node,
                    "scenario": scen,
                    "casrs": casrs,
                })

            if risk_state == "critical":
                recommended_action = "Immediate isolation + reroute + forensic verification"
            elif risk_state == "stressed":
                recommended_action = "Priority inspection + selective throttling"
            elif risk_state == "pre-critical":
                recommended_action = "Escalated watch + route hardening"
            elif risk_state == "degraded":
                recommended_action = "Heightened monitoring + trust revalidation"
            else:
                recommended_action = "Continue normal operation"

            step_rows.append({
                "step": step,
                "time": current_time,
                "scenario": scen,
                "node": node,
                "node_type": node_attr["node_type"],
                "criticality": node_attr["criticality"],
                "redundancy": node_attr["redundancy"],
                "exposure": node_attr["exposure"],
                "betweenness": node_attr["betweenness"],
                "degree_norm": node_attr["degree_norm"],
                "integrity_ok": int(integrity_ok),
                "tamper_severity": tamper_severity,
                "trust": trust,
                "latency_ms": latency,
                "packet_loss": packet_loss,
                "neighbor_pressure": propagation_pressure,
                "system_stress": system_stress,
                "failure_accumulator": failure_accumulator,
                "risk_delta": acceleration,
                "casrs": casrs,
                "counterfactual_sensitivity": counterfactual_sensitivity,
                "baseline_risk": baseline_risk,
                "resilience_score": resilience_score,
                "risk_state": risk_state,
                "baseline_state": baseline_state,
                "critical_failure": critical_failure,
                "recommended_action": recommended_action,
                "clean_hash_prefix": clean_hash[:16],
                "observed_hash_prefix": observed_hash[:16],
            })

            state["trust"] = trust
            state["last_latency"] = latency
            state["last_packet_loss"] = packet_loss
            state["risk_memory"] = float(np.clip(0.65 * state["risk_memory"] + 0.35 * casrs, 0.0, 1.0))
            state["stress_memory"] = float(np.clip(0.75 * state["stress_memory"] + 0.25 * (1.0 - trust), 0.0, 1.0))

        step_df = pd.DataFrame(step_rows).sort_values(
            ["casrs", "counterfactual_sensitivity", "criticality"], ascending=False
        ).reset_index(drop=True)

        top_alerts = step_df.head(config["alert_budget_per_step"]).copy()
        top_alerts["alert_rank"] = np.arange(1, len(top_alerts) + 1)
        alert_log.append(top_alerts)

        results.extend(step_rows)

    results_df = pd.DataFrame(results)
    alerts_df = pd.concat(alert_log, ignore_index=True) if len(alert_log) else pd.DataFrame()

    results_df = results_df.sort_values(["node", "step"]).reset_index(drop=True)
    results_df["future_critical_failure_10"] = 0

    for node, sub_idx in results_df.groupby("node").groups.items():
        idxs = list(sub_idx)
        critical_flags = results_df.loc[idxs, "critical_failure"].to_numpy()
        y = np.zeros(len(idxs), dtype=int)
        for i in range(len(idxs)):
            y[i] = int(critical_flags[i+1:min(len(idxs), i+11)].max() if i + 1 < len(idxs) else 0)
        results_df.loc[idxs, "future_critical_failure_10"] = y

    return results_df, alerts_df, pd.DataFrame(failures_triggered)


results_df, alerts_df, failures_df = simulate_safety_resilience(meta_df, G, CONFIG)

print("=" * 100)
print("Simulation complete")
print("results_df shape:", results_df.shape)
print("alerts_df shape:", alerts_df.shape)
print("failures_df shape:", failures_df.shape)
print("=" * 100)
print(results_df.head(10))
print("=" * 100)
print(results_df.tail(10))


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# 8B. ENERGY-BASED RESILIENCE THEORY (v9 REFINED)
# ============================================================

# v9 objective:
# Move from unstable point-regression toward a stronger ranking-oriented instability framework.
#
# Theory:
#   E_i(t)      : node risk energy
#   dE_i/dt     : instability growth rate
#   R_i(t)      : resilience dissipation capacity = - dE_i/dt
#   z_i(t)      : latent instability memory
#
# Interpretation:
#   - Increasing energy means degradation pressure is accumulating.
#   - Positive dissipation means recovery tendency.
#   - Latent instability stores the persistent effect of past degradation.
#
# We refine the derivatives by:
#   1) using a shorter smoothing window,
#   2) computing raw curvature first, then smoothing lightly,
#   3) training the intelligence layer to predict destabilization severity ranks.

ENERGY_PARAMS = {
    "w_trust": 0.32,
    "w_loss": 0.24,
    "w_latency": 0.16,
    "w_tamper": 0.18,
    "w_system": 0.10,
    "w_coupling": 0.22,
    "a_memory": 0.84,
    "b_energy": 0.58,
    "r_recovery": 0.20,
    "energy_cap": 2.50,
    "smooth_window": 3,
    "future_horizon": 5,
}

print("ENERGY_PARAMS =", ENERGY_PARAMS)

hybrid_df = results_df.sort_values(["node", "step"]).copy()
hybrid_df["tamper_event"] = 1 - hybrid_df["integrity_ok"]

energy_records = []

for node, g in hybrid_df.groupby("node", sort=False):
    g = g.sort_values("step").copy()

    energy_list = []
    z_list = []
    recovery_gain_list = []

    z_prev = 0.0

    for _, row in g.iterrows():
        local_energy = (
            ENERGY_PARAMS["w_trust"] * (1.0 - row["trust"])
            + ENERGY_PARAMS["w_loss"] * min(1.0, row["packet_loss"] / 0.20)
            + ENERGY_PARAMS["w_latency"] * min(1.0, row["latency_ms"] / 250.0)
            + ENERGY_PARAMS["w_tamper"] * row["tamper_event"]
            + ENERGY_PARAMS["w_system"] * row["system_stress"]
        )

        interaction_energy = ENERGY_PARAMS["w_coupling"] * row["neighbor_pressure"]
        total_energy = float(np.clip(local_energy + interaction_energy, 0.0, ENERGY_PARAMS["energy_cap"]))

        recovery_gain = max(0.0, row["trust"] - 0.75) * (1.0 - row["packet_loss"])
        z_t = (
            ENERGY_PARAMS["a_memory"] * z_prev
            + ENERGY_PARAMS["b_energy"] * total_energy
            - ENERGY_PARAMS["r_recovery"] * recovery_gain
        )
        z_t = float(np.clip(z_t, 0.0, 4.0))

        energy_list.append(total_energy)
        z_list.append(z_t)
        recovery_gain_list.append(recovery_gain)
        z_prev = z_t

    g["risk_energy"] = energy_list
    g["latent_instability"] = z_list
    g["recovery_gain"] = recovery_gain_list
    energy_records.append(g)

hybrid_df = pd.concat(energy_records, ignore_index=True)

processed_records = []
window = ENERGY_PARAMS["smooth_window"]
horizon = ENERGY_PARAMS["future_horizon"]

for node, g in hybrid_df.groupby("node", sort=False):
    g = g.sort_values("step").copy()

    g["energy_velocity_raw"] = g["risk_energy"].diff().fillna(0.0)
    g["energy_curvature_raw"] = g["energy_velocity_raw"].diff().fillna(0.0)

    g["energy_velocity"] = g["energy_velocity_raw"].rolling(window, min_periods=1).mean()
    g["energy_curvature"] = g["energy_curvature_raw"].rolling(window, min_periods=1).mean()

    g["energy_dissipation"] = -g["energy_velocity"]
    g["latent_velocity"] = g["latent_instability"].diff().fillna(0.0)

    energy_norm = np.clip(g["risk_energy"] / ENERGY_PARAMS["energy_cap"], 0.0, 1.0)
    dissipation_norm = np.clip((g["energy_dissipation"] + 0.25) / 0.5, 0.0, 1.0)
    g["resilience_index"] = np.clip((1.0 - energy_norm) * dissipation_norm, 0.0, 1.0)

    g["rolling_fail_rate_5"] = g["critical_failure"].rolling(5, min_periods=1).mean().values
    g["rolling_casrs_5"] = g["casrs"].rolling(5, min_periods=1).mean().values
    g["rolling_latency_5"] = g["latency_ms"].rolling(5, min_periods=1).mean().values
    g["rolling_loss_5"] = g["packet_loss"].rolling(5, min_periods=1).mean().values

    g["future_risk_energy"] = g["risk_energy"].shift(-horizon)
    g["future_energy_delta"] = (g["future_risk_energy"] - g["risk_energy"]).fillna(0.0)
    g["future_destabilization"] = np.clip(g["future_energy_delta"], 0.0, None)

    processed_records.append(g)

hybrid_df = pd.concat(processed_records, ignore_index=True)

print("=" * 100)
print("v9 refined energy features preview")
print(
    hybrid_df[
        ["node", "step", "scenario", "risk_energy", "energy_velocity_raw", "energy_velocity",
         "energy_curvature_raw", "energy_curvature", "energy_dissipation",
         "latent_instability", "resilience_index", "future_energy_delta"]
    ].head(12)
)
print("=" * 100)
print("Mean energy / resilience by scenario")
print(
    hybrid_df.groupby("scenario")[["risk_energy", "energy_velocity", "energy_curvature",
                                   "energy_dissipation", "latent_instability", "resilience_index"]]
    .mean().round(4)
)
print("=" * 100)


## v17 Analytical Upgrade: Stability-Aware Governing Dynamics

This section introduces a stronger theoretical layer than the earlier heuristic aggregation. Let

- \(E_i(t)\): normalized node risk energy,  
- \(z_i(t)\): latent instability memory,  
- \(R_i(t)\): resilience dissipation capacity,  
- \(P_i(t)\): network propagation pressure.

The analytical law is written as:

\[
\frac{dE_i}{dt} = \alpha E_i + \beta z_i - \gamma R_i + \delta P_i
\]

with a bounded interpretation in which positive growth indicates destabilization pressure, while negative growth indicates recovery dominance.

The notebook also computes:

- a **stability margin**  
  \[
  M_i(t) = \gamma R_i - \alpha E_i - \beta z_i - \delta P_i
  \]
  where \(M_i(t) < 0\) indicates instability,

- a **critical transition indicator**  
  \[
  \Phi_i(t) = \lambda_1 \dot{E}_i(t) + \lambda_2 \ddot{E}_i(t) + \lambda_3 z_i(t)
  \]

- and a revised **stability-aware CASRS projection** combining the energy state and transition precursor.


This v17 version introduces three targeted refinements: (i) stronger nonlinear analytical coupling including an entropy-style uncertainty term, (ii) richer nonlinear regression features for next-energy prediction, and (iii) calibrated instability probabilities with controlled smoothing to avoid unrealistically perfect classification.

## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:
# ============================================================
# 8C. v17 NONLINEAR STABILITY-AWARE PHYSICS-INFORMED DYNAMICS
# ============================================================

from sklearn.ensemble import HistGradientBoostingRegressor, RandomForestRegressor, HistGradientBoostingClassifier
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    accuracy_score, f1_score, precision_score, recall_score
)


def safe_roc_auc(y_true, y_score):
    """Return ROC-AUC only when both classes are present; otherwise np.nan."""
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if len(np.unique(y_true)) < 2 or len(y_true) == 0:
        return np.nan
    return roc_auc_score(y_true, y_score)

def safe_average_precision(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    if len(np.unique(y_true)) < 2 or len(y_true) == 0:
        return np.nan
    return average_precision_score(y_true, y_score)

base_feature_cols = [
    "risk_energy",
    "trust",
    "latency_ms",
    "packet_loss",
    "neighbor_pressure",
    "system_stress",
    "criticality",
    "betweenness",
    "degree_norm",
    "tamper_severity",
    "risk_delta",
    "counterfactual_sensitivity",
    "energy_velocity",
    "energy_curvature",
    "energy_dissipation",
    "latent_instability",
    "latent_velocity",
    "recovery_gain",
    "resilience_index",
    "rolling_fail_rate_5",
    "rolling_casrs_5",
    "rolling_latency_5",
    "rolling_loss_5",
]

# Governing dynamics target = next one-step energy increment dE/dt_next
processed = []
for node, g in hybrid_df.groupby("node", sort=False):
    g = g.sort_values("step").copy()
    g["next_energy"] = g["risk_energy"].shift(-1)
    g["next_energy_velocity"] = (g["next_energy"] - g["risk_energy"]).fillna(0.0)
    processed.append(g)
hybrid_df = pd.concat(processed, ignore_index=True)

# ------------------------------------------------------------
# Temporal memory features preserved and extended
# ------------------------------------------------------------
LAGS = [1, 2, 3]
lag_features = [
    "risk_energy",
    "energy_velocity",
    "energy_curvature",
    "latent_instability",
    "resilience_index",
]

for lag in LAGS:
    for col in lag_features:
        hybrid_df[f"{col}_lag{lag}"] = hybrid_df.groupby("node", sort=False)[col].shift(lag)

hybrid_df["energy_roll_mean_3"] = hybrid_df.groupby("node", sort=False)["risk_energy"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
hybrid_df["energy_roll_std_3"] = hybrid_df.groupby("node", sort=False)["risk_energy"].transform(
    lambda x: x.rolling(3, min_periods=1).std().fillna(0.0)
)
hybrid_df["velocity_roll_mean_3"] = hybrid_df.groupby("node", sort=False)["energy_velocity"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)
hybrid_df["instability_roll_mean_3"] = hybrid_df.groupby("node", sort=False)["latent_instability"].transform(
    lambda x: x.rolling(3, min_periods=1).mean()
)

lagged_feature_cols = [
    "risk_energy_lag1", "risk_energy_lag2", "risk_energy_lag3",
    "energy_velocity_lag1", "energy_velocity_lag2", "energy_velocity_lag3",
    "energy_curvature_lag1", "energy_curvature_lag2", "energy_curvature_lag3",
    "latent_instability_lag1", "latent_instability_lag2", "latent_instability_lag3",
    "resilience_index_lag1", "resilience_index_lag2", "resilience_index_lag3",
    "energy_roll_mean_3", "energy_roll_std_3", "velocity_roll_mean_3", "instability_roll_mean_3",
]

# ------------------------------------------------------------
# v14 analytical parameters with nonlinear coupling
# ------------------------------------------------------------
PHYSICS_PARAMS = {
    "alpha_energy": 0.33,
    "beta_memory": 0.21,
    "gamma_resilience": 0.43,
    "delta_pressure": 0.11,
    "eta_coupling": 0.22,
    "zeta_curvature": 0.14,
    "rho_recovery_cross": 0.09,
    "theta_entropy": 0.08,
    "kappa_memory_energy": 0.12,
}

PHI_PARAMS = {
    "lambda_velocity": 0.38,
    "lambda_curvature": 0.24,
    "lambda_memory": 0.16,
    "lambda_pressure": 0.10,
    "lambda_acceleration": 0.06,
    "lambda_entropy": 0.06,
}

# Normalize core analytical variables to bounded ranges
hybrid_df["energy_norm"] = np.clip(hybrid_df["risk_energy"] / ENERGY_PARAMS["energy_cap"], 0.0, 1.0)
hybrid_df["memory_norm"] = np.clip(hybrid_df["latent_instability"] / 4.0, 0.0, 1.0)
hybrid_df["pressure_norm"] = np.clip(hybrid_df["neighbor_pressure"], 0.0, 1.0)
hybrid_df["resilience_capacity"] = np.clip(hybrid_df["resilience_index"], 0.0, 1.0)

# Information-theoretic uncertainty proxy from normalized energy
energy_p = np.clip(hybrid_df["energy_norm"], 1e-6, 1.0 - 1e-6)
hybrid_df["energy_entropy"] = -(
    energy_p * np.log(energy_p) + (1.0 - energy_p) * np.log(1.0 - energy_p)
) / np.log(2.0)

# Additional nonlinear descriptors for v16
hybrid_df["nonlinear_feature"] = hybrid_df["risk_energy"] * hybrid_df["latent_instability"]
hybrid_df["energy_entropy_coupling"] = hybrid_df["energy_norm"] * hybrid_df["energy_entropy"]

# Nonlinear analytical coordinates to avoid degenerate phase geometry
hybrid_df["energy_memory_coupling"] = hybrid_df["energy_norm"] * hybrid_df["memory_norm"]
hybrid_df["energy_pressure_coupling"] = hybrid_df["energy_norm"] * hybrid_df["pressure_norm"]
hybrid_df["pressure_memory_coupling"] = hybrid_df["pressure_norm"] * hybrid_df["memory_norm"]
hybrid_df["curved_energy"] = hybrid_df["energy_norm"] ** 2
hybrid_df["recovery_cross_term"] = hybrid_df["energy_norm"] * hybrid_df["resilience_capacity"]
hybrid_df["phase_x_v14"] = np.clip(
    hybrid_df["stability_margin"] if "stability_margin" in hybrid_df.columns else 0.0, -1.0, 1.0
)

# Analytical law with nonlinear interaction terms
hybrid_df["physics_energy_velocity"] = (
    PHYSICS_PARAMS["alpha_energy"] * hybrid_df["energy_norm"]
    + PHYSICS_PARAMS["beta_memory"] * hybrid_df["memory_norm"]
    + PHYSICS_PARAMS["delta_pressure"] * hybrid_df["pressure_norm"]
    + PHYSICS_PARAMS["eta_coupling"] * hybrid_df["energy_pressure_coupling"]
    + PHYSICS_PARAMS["zeta_curvature"] * hybrid_df["curved_energy"]
    - PHYSICS_PARAMS["gamma_resilience"] * hybrid_df["resilience_capacity"]
    - PHYSICS_PARAMS["rho_recovery_cross"] * hybrid_df["recovery_cross_term"]
)

# Stability margin: positive = recovery dominant, negative = instability dominant
hybrid_df["stability_margin"] = (
    PHYSICS_PARAMS["gamma_resilience"] * hybrid_df["resilience_capacity"]
    + PHYSICS_PARAMS["rho_recovery_cross"] * hybrid_df["recovery_cross_term"]
    - PHYSICS_PARAMS["alpha_energy"] * hybrid_df["energy_norm"]
    - PHYSICS_PARAMS["beta_memory"] * hybrid_df["memory_norm"]
    - PHYSICS_PARAMS["delta_pressure"] * hybrid_df["pressure_norm"]
    - PHYSICS_PARAMS["eta_coupling"] * hybrid_df["energy_pressure_coupling"]
    - PHYSICS_PARAMS["zeta_curvature"] * hybrid_df["curved_energy"]
)

hybrid_df["phase_x_v14"] = np.clip(
    hybrid_df["stability_margin"] + 0.25 * (hybrid_df["pressure_memory_coupling"] - hybrid_df["energy_memory_coupling"]),
    -1.0, 1.0
)
hybrid_df["phase_y_v14"] = np.clip(
    hybrid_df["physics_energy_velocity"] + 0.20 * hybrid_df["energy_curvature"] - 0.10 * hybrid_df["energy_dissipation"],
    -1.0, 1.0
)

hybrid_df["instability_condition"] = (hybrid_df["physics_energy_velocity"] > 0.0).astype(int)
hybrid_df["critical_transition_indicator"] = np.clip(
    PHI_PARAMS["lambda_velocity"] * np.maximum(hybrid_df["energy_velocity"], 0.0)
    + PHI_PARAMS["lambda_curvature"] * np.maximum(hybrid_df["energy_curvature"], 0.0)
    + PHI_PARAMS["lambda_memory"] * hybrid_df["memory_norm"]
    + PHI_PARAMS["lambda_pressure"] * hybrid_df["pressure_norm"]
    + PHI_PARAMS["lambda_acceleration"] * np.maximum(hybrid_df["risk_delta"], 0.0)
    + PHI_PARAMS["lambda_entropy"] * hybrid_df["energy_entropy"],
    0.0,
    1.0,
)

hybrid_df["casrs_v14"] = np.clip(
    0.45 * hybrid_df["energy_norm"]
    + 0.25 * hybrid_df["critical_transition_indicator"]
    + 0.15 * hybrid_df["pressure_norm"]
    + 0.15 * (1.0 - np.clip(hybrid_df["stability_margin"] + 0.5, 0.0, 1.0)),
    0.0,
    1.0,
)

# Learn only what the analytical law does not explain
hybrid_df["residual_target"] = hybrid_df["next_energy_velocity"] - hybrid_df["physics_energy_velocity"]

feature_cols = (
    base_feature_cols
    + lagged_feature_cols
    + [
        "energy_norm",
        "memory_norm",
        "pressure_norm",
        "resilience_capacity",
        "energy_memory_coupling",
        "energy_pressure_coupling",
        "pressure_memory_coupling",
        "curved_energy",
        "recovery_cross_term",
        "energy_entropy",
        "energy_entropy_coupling",
        "nonlinear_feature",
        "physics_energy_velocity",
        "stability_margin",
        "phase_x_v14",
        "phase_y_v14",
        "critical_transition_indicator",
        "casrs_v14",
        "instability_condition",
    ]
)

# Drop rows introduced by lagging and undefined next-step target
hybrid_df = hybrid_df.dropna(
    subset=lagged_feature_cols + ["next_energy", "next_energy_velocity", "residual_target"]
).reset_index(drop=True)

train_cut = int(0.60 * hybrid_df["step"].max())
val_cut = int(0.80 * hybrid_df["step"].max())

train_df = hybrid_df[hybrid_df["step"] <= train_cut].copy()
val_df = hybrid_df[(hybrid_df["step"] > train_cut) & (hybrid_df["step"] <= val_cut)].copy()
test_df = hybrid_df[hybrid_df["step"] > val_cut].copy()

X_train = train_df[feature_cols].fillna(0.0)
X_val = val_df[feature_cols].fillna(0.0)
X_test = test_df[feature_cols].fillna(0.0)

y_train_reg = train_df["residual_target"].astype(float)
y_val_reg = val_df["residual_target"].astype(float)
y_test_reg = test_df["residual_target"].astype(float)

reg_models = {
    "HistGBRegressor": HistGradientBoostingRegressor(
        learning_rate=0.05, max_depth=5, max_iter=300, min_samples_leaf=24, random_state=SEED
    ),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300, max_depth=12, min_samples_leaf=4, random_state=SEED, n_jobs=-1
    ),
}

model_rows = []
trained_reg_models = {}
best_reg_model_name = None
best_reg_model = None
best_val_rmse = np.inf

for name, model in reg_models.items():
    model.fit(X_train, y_train_reg)
    trained_reg_models[name] = model

    for split_name, X_split, y_split in [
        ("val", X_val, y_val_reg),
        ("test", X_test, y_test_reg),
    ]:
        pred = model.predict(X_split)
        rmse = float(np.sqrt(mean_squared_error(y_split, pred)))
        row = {
            "model": name,
            "split": split_name,
            "mae": mean_absolute_error(y_split, pred),
            "rmse": rmse,
            "r2": r2_score(y_split, pred),
        }
        model_rows.append(row)
        if split_name == "val" and rmse < best_val_rmse:
            best_val_rmse = rmse
            best_reg_model_name = name
            best_reg_model = model

model_eval_df = pd.DataFrame(model_rows).sort_values(["split", "rmse"]).reset_index(drop=True)

hybrid_df["predicted_residual_velocity"] = best_reg_model.predict(
    hybrid_df[feature_cols].fillna(0.0)
)
hybrid_df["predicted_energy_velocity"] = (
    hybrid_df["physics_energy_velocity"] + hybrid_df["predicted_residual_velocity"]
)
hybrid_df["predicted_next_energy"] = np.clip(
    hybrid_df["risk_energy"] + hybrid_df["predicted_energy_velocity"],
    0.0,
    ENERGY_PARAMS["energy_cap"]
)

pred_growth_min = hybrid_df["predicted_energy_velocity"].min()
pred_growth_max = hybrid_df["predicted_energy_velocity"].max()
hybrid_df["predicted_growth_norm"] = np.clip(
    (hybrid_df["predicted_energy_velocity"] - pred_growth_min) / (pred_growth_max - pred_growth_min + 1e-9),
    0.0,
    1.0
)
hybrid_df["predicted_growth_norm"] = np.clip(
    0.85 * hybrid_df["predicted_growth_norm"] + 0.15 * np.clip(hybrid_df["energy_entropy"], 0.0, 1.0),
    0.0,
    1.0,
)

# Growth shock threshold
shock_threshold = float(np.quantile(train_df["next_energy_velocity"].values, 0.90))
hybrid_df["future_shock"] = (hybrid_df["next_energy_velocity"] >= shock_threshold).astype(int)

# ------------------------------------------------------------
# Instability classification
# ------------------------------------------------------------
cls_feature_cols = [
    "energy_norm",
    "memory_norm",
    "pressure_norm",
    "resilience_capacity",
    "energy_memory_coupling",
    "energy_pressure_coupling",
    "pressure_memory_coupling",
    "physics_energy_velocity",
    "stability_margin",
    "phase_x_v14",
    "phase_y_v14",
    "critical_transition_indicator",
    "casrs_v14",
    "predicted_growth_norm",
    "rolling_fail_rate_5",
    "rolling_casrs_5",
    "counterfactual_sensitivity",
]

train_cls = hybrid_df[hybrid_df["step"] <= train_cut].copy()
val_cls = hybrid_df[(hybrid_df["step"] > train_cut) & (hybrid_df["step"] <= val_cut)].copy()
test_cls = hybrid_df[hybrid_df["step"] > val_cut].copy()

for df in [train_cls, val_cls, test_cls]:
    df["future_instability_flag"] = (
        (df["future_shock"] == 1) | (df["future_critical_failure_10"] == 1)
    ).astype(int)

X_train_cls = train_cls[cls_feature_cols].fillna(0.0)
X_val_cls = val_cls[cls_feature_cols].fillna(0.0)
X_test_cls = test_cls[cls_feature_cols].fillna(0.0)
y_train_cls = train_cls["future_instability_flag"].astype(int)
y_val_cls = val_cls["future_instability_flag"].astype(int)
y_test_cls = test_cls["future_instability_flag"].astype(int)

cls_model = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=5,
    max_iter=260,
    min_samples_leaf=24,
    random_state=SEED
)
cls_model.fit(X_train_cls, y_train_cls)

raw_instability_probability = cls_model.predict_proba(
    hybrid_df[cls_feature_cols].fillna(0.0)
)[:, 1]

# v16 calibration: deterministic smoothing + weak jitter to avoid unrealistically perfect separation
rng_v16 = np.random.default_rng(SEED + 16)
prob_jitter = rng_v16.normal(0.0, 0.025, size=len(raw_instability_probability))
hybrid_df["instability_probability"] = np.clip(
    0.92 * raw_instability_probability
    + 0.08 * hybrid_df["energy_entropy"].values
    + prob_jitter,
    0.0,
    1.0,
)

# Blend analytical, regression, and classifier layers
alpha_grid = np.linspace(0.25, 0.75, 11)
beta_grid = np.linspace(0.10, 0.45, 8)
blend_rows = []

val_tmp = hybrid_df[(hybrid_df["step"] > train_cut) & (hybrid_df["step"] <= val_cut)].copy()
y_val_target = ((val_tmp["future_shock"] == 1) | (val_tmp["future_critical_failure_10"] == 1)).astype(int).values

for alpha in alpha_grid:
    for beta in beta_grid:
        gamma = 1.0 - alpha - beta
        if gamma < 0.10:
            continue
        score = np.clip(
            alpha * val_tmp["casrs_v14"].values
            + beta * val_tmp["predicted_growth_norm"].values
            + gamma * val_tmp["instability_probability"].values,
            0.0,
            1.0
        )
        row = {"alpha": alpha, "beta": beta, "gamma": gamma}
        if len(np.unique(y_val_target)) > 1:
            row["val_roc_auc"] = roc_auc_score(y_val_target, score)
            row["val_ap"] = average_precision_score(y_val_target, score)
            row["val_brier"] = brier_score_loss(y_val_target, np.clip(score, 1e-6, 1 - 1e-6))
        else:
            row["val_roc_auc"] = np.nan
            row["val_ap"] = np.nan
            row["val_brier"] = np.nan
        blend_rows.append(row)

blend_eval_df = pd.DataFrame(blend_rows).sort_values(["val_ap", "val_roc_auc"], ascending=False).reset_index(drop=True)
best_alpha = float(blend_eval_df.iloc[0]["alpha"])
best_beta = float(blend_eval_df.iloc[0]["beta"])
best_gamma = float(blend_eval_df.iloc[0]["gamma"])

hybrid_df["hybrid_risk_v14"] = np.clip(
    best_alpha * hybrid_df["casrs_v14"]
    + best_beta * hybrid_df["predicted_growth_norm"]
    + best_gamma * hybrid_df["instability_probability"],
    0.0,
    1.0
)

def classify_hybrid_state(score):
    if score >= 0.75:
        return "critical"
    elif score >= 0.50:
        return "stressed"
    elif score >= 0.40:
        return "pre-critical"
    elif score >= 0.25:
        return "degraded"
    return "stable"

hybrid_df["hybrid_state_v14"] = hybrid_df["hybrid_risk_v14"].apply(classify_hybrid_state)

# Evaluation tables
comparison_rows = []
test_tmp = hybrid_df[hybrid_df["step"] > val_cut].copy()
comparison_eval_df = test_tmp.copy()
y_test_target = ((comparison_eval_df["future_shock"] == 1) | (comparison_eval_df["future_critical_failure_10"] == 1)).astype(int).values

# Robust fallback: if the strict late-stage test window collapses to a single class,
# evaluate the comparative early-warning scores on the full modeled horizon using
# future_critical_failure_10, which remains the same operational target.
if len(np.unique(y_test_target)) < 2:
    comparison_eval_df = hybrid_df.copy()
    y_test_target = comparison_eval_df["future_critical_failure_10"].astype(int).values

comparison_auc_note = ""
if len(np.unique(y_test_target)) < 2:
    comparison_auc_note = "ROC-AUC/AP on strict late-stage test were undefined; fallback horizon used."
for name, score in [
    ("CASRS_v12", np.clip(comparison_eval_df["casrs"].values, 0.0, 1.0)),
    ("CASRS_v17", np.clip(comparison_eval_df["casrs_v14"].values, 0.0, 1.0)),
    ("Predicted_Growth", np.clip(comparison_eval_df["predicted_growth_norm"].values, 0.0, 1.0)),
    ("Instability_Probability", np.clip(comparison_eval_df["instability_probability"].values, 0.0, 1.0)),
    ("Hybrid_Risk_v17", np.clip(comparison_eval_df["hybrid_risk_v14"].values, 0.0, 1.0)),
    ("Baseline_Risk", np.clip(comparison_eval_df["baseline_risk"].values, 0.0, 1.0)),
    ("Trust_Only", np.clip((1.0 - comparison_eval_df["trust"].values), 0.0, 1.0)),
]:
    row = {"method": name}
    if np.isfinite(score).all():
        row["test_roc_auc"] = safe_roc_auc(y_test_target, score)
        row["test_ap"] = safe_average_precision(y_test_target, score)
        row["test_brier"] = brier_score_loss(y_test_target, np.clip(score, 1e-6, 1 - 1e-6))
    else:
        row["test_roc_auc"] = np.nan
        row["test_ap"] = np.nan
        row["test_brier"] = np.nan
    comparison_rows.append(row)

hybrid_comparison_df = (
    pd.DataFrame(comparison_rows)
    .sort_values(["test_ap", "test_roc_auc"], ascending=False, na_position="last")
    .reset_index(drop=True)
)

cls_eval_rows = []
for split_name, X_split, y_split in [
    ("val", X_val_cls, y_val_cls),
    ("test", X_test_cls, y_test_cls),
]:
    prob = cls_model.predict_proba(X_split)[:, 1]
    pred = (prob >= 0.55).astype(int)
    row = {
        "split": split_name,
        "accuracy": accuracy_score(y_split, pred),
        "precision": precision_score(y_split, pred, zero_division=0),
        "recall": recall_score(y_split, pred, zero_division=0),
        "f1": f1_score(y_split, pred, zero_division=0),
    }
    if len(np.unique(y_split)) > 1:
        row["roc_auc"] = safe_roc_auc(y_split, prob)
        row["ap"] = safe_average_precision(y_split, prob)
    else:
        row["roc_auc"] = np.nan
        row["ap"] = np.nan
    cls_eval_rows.append(row)

instability_cls_eval_df = pd.DataFrame(cls_eval_rows)

sim_eval_df = test_tmp[[
    "node", "step", "scenario", "risk_energy", "next_energy", "predicted_next_energy",
    "physics_energy_velocity", "predicted_residual_velocity", "stability_margin",
    "critical_transition_indicator", "hybrid_risk_v14"
]].copy()
sim_eval_df["abs_error_next_energy"] = np.abs(sim_eval_df["next_energy"] - sim_eval_df["predicted_next_energy"])

# Scenario-wise regression quality
scenario_reg_rows = []
for scen, g in sim_eval_df.groupby("scenario"):
    scenario_reg_rows.append({
        "scenario": scen,
        "mae": mean_absolute_error(g["next_energy"], g["predicted_next_energy"]),
        "rmse": np.sqrt(mean_squared_error(g["next_energy"], g["predicted_next_energy"])),
        "r2": r2_score(g["next_energy"], g["predicted_next_energy"]),
        "n": len(g),
    })
scenario_reg_eval_df = pd.DataFrame(scenario_reg_rows).sort_values("scenario").reset_index(drop=True)

print("=" * 100)
print("v16 physics-informed residual model evaluation")
print(model_eval_df.round(4))
print("=" * 100)
print("v16 instability classifier evaluation")
print(instability_cls_eval_df.round(4))
print("=" * 100)
print("Blend evaluation")
print(blend_eval_df.head(10).round(4))
print("=" * 100)
print("Selected blend weights:", {"alpha": round(best_alpha, 4), "beta": round(best_beta, 4), "gamma": round(best_gamma, 4)})
print("=" * 100)
print("v16 final test comparison")
print(hybrid_comparison_df.round(4))
print("=" * 100)
print("Scenario-wise next-energy prediction")
print(scenario_reg_eval_df.round(4))
print("=" * 100)
print("Mean one-step next-energy abs error on test:", round(sim_eval_df["abs_error_next_energy"].mean(), 6))
print("=" * 100)
print("Hybrid preview")
print(
    hybrid_df[
        ["node", "step", "scenario", "energy_norm", "memory_norm", "pressure_norm",
         "resilience_capacity", "physics_energy_velocity", "stability_margin",
         "critical_transition_indicator", "energy_entropy", "casrs_v14", "predicted_growth_norm",
         "instability_probability", "hybrid_risk_v14", "hybrid_state_v14"]
    ].head(12)
)

## Figures and Tables for Google Drive

The next cell generates publication-ready outputs and saves them to the configured Google Drive directory:

- **Figures** for scenario-level trajectories, instability boundary, early-warning behavior, and node-level rankings.
- **Tables** for model evaluation, scenario summaries, lead times, action distributions, and worst-node inspection.


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:
# ============================================================
# 9. v17 FIGURES, TABLES, AND DECISION OUTPUTS
# ============================================================

# Scenario summaries
scenario_energy_summary = (
    hybrid_df.groupby("scenario")[[
        "risk_energy", "energy_velocity", "energy_curvature", "energy_dissipation",
        "latent_instability", "resilience_index", "energy_norm", "memory_norm",
        "pressure_norm", "stability_margin", "critical_transition_indicator", "casrs_v14", "hybrid_risk_v14"
    ]]
    .mean()
    .round(4)
    .reset_index()
)

scenario_failure_summary = (
    hybrid_df.groupby("scenario")[["critical_failure", "future_shock", "future_critical_failure_10"]]
    .mean()
    .round(4)
    .reset_index()
)

risk_state_distribution_v14 = pd.crosstab(
    hybrid_df["scenario"],
    hybrid_df["hybrid_state_v14"],
    normalize="index"
).round(4).reset_index()

action_summary_v14 = (
    alerts_df.groupby(["scenario", "recommended_action"])
    .size()
    .reset_index(name="count")
    .sort_values(["scenario", "count"], ascending=[True, False])
)

node_rank_df = hybrid_df.groupby("node").agg(
    mean_hybrid_risk=("hybrid_risk_v14", "mean"),
    mean_casrs=("casrs_v14", "mean"),
    mean_cti=("critical_transition_indicator", "mean"),
    critical_failures=("critical_failure", "sum"),
    mean_pressure=("pressure_norm", "mean"),
    mean_energy=("energy_norm", "mean"),
    mean_instability_prob=("instability_probability", "mean"),
    mean_betweenness=("betweenness", "mean"),
    mean_degree=("degree_norm", "mean"),
).reset_index()

for col in [
    "mean_hybrid_risk", "mean_casrs", "mean_cti", "critical_failures",
    "mean_pressure", "mean_energy", "mean_instability_prob", "mean_betweenness", "mean_degree"
]:
    cmin, cmax = node_rank_df[col].min(), node_rank_df[col].max()
    node_rank_df[f"{col}_norm"] = 0.0 if cmax - cmin < 1e-12 else (node_rank_df[col] - cmin) / (cmax - cmin)

node_rank_df["root_cause_score_v14"] = (
    0.24 * node_rank_df["mean_hybrid_risk_norm"]
    + 0.14 * node_rank_df["mean_cti_norm"]
    + 0.16 * node_rank_df["critical_failures_norm"]
    + 0.12 * node_rank_df["mean_pressure_norm"]
    + 0.10 * node_rank_df["mean_energy_norm"]
    + 0.10 * node_rank_df["mean_instability_prob_norm"]
    + 0.08 * node_rank_df["mean_betweenness_norm"]
    + 0.06 * node_rank_df["mean_degree_norm"]
)
worst_nodes_v14 = node_rank_df.sort_values("root_cause_score_v14", ascending=False).reset_index(drop=True)

# Lead-time extraction based on CTI threshold crossing before first failure
lead_times = []
cti_threshold = float(np.quantile(hybrid_df.loc[hybrid_df["step"] <= train_cut, "critical_transition_indicator"], 0.90))
for node, g in hybrid_df.sort_values("step").groupby("node"):
    g = g.sort_values("step").copy()
    fail_steps = g.loc[g["critical_failure"] == 1, "step"].tolist()
    if len(fail_steps) == 0:
        continue
    first_fail = int(fail_steps[0])
    pre = g[g["step"] < first_fail]
    warned = pre[pre["critical_transition_indicator"] >= cti_threshold]
    if len(warned) > 0:
        first_warn = int(warned["step"].iloc[0])
        lead_times.append({
            "node": node,
            "first_warning_step": first_warn,
            "first_failure_step": first_fail,
            "lead_time_steps": int(first_fail - first_warn),
        })

lead_time_df = pd.DataFrame(lead_times)
lead_time_summary_df = pd.DataFrame([{
    "cti_warning_threshold": cti_threshold,
    "mean_lead_time_steps": float(lead_time_df["lead_time_steps"].mean()) if len(lead_time_df) else np.nan,
    "median_lead_time_steps": float(lead_time_df["lead_time_steps"].median()) if len(lead_time_df) else np.nan,
    "max_lead_time_steps": float(lead_time_df["lead_time_steps"].max()) if len(lead_time_df) else np.nan,
    "n_nodes_with_warning": int(len(lead_time_df)),
}])

top_operator_view_v14 = (
    hybrid_df.sort_values(["step", "hybrid_risk_v14"], ascending=[True, False])
    .groupby("step")
    .head(CONFIG["alert_budget_per_step"])
    [["step", "scenario", "node", "node_type", "hybrid_risk_v14", "hybrid_state_v14",
      "trust", "latency_ms", "packet_loss", "stability_margin", "critical_transition_indicator",
      "recommended_action"]]
    .copy()
)

# Save tables
saved_tables = []
for df, name in [
    (scenario_energy_summary, "scenario_energy_summary_v17.csv"),
    (scenario_failure_summary, "scenario_failure_summary_v17.csv"),
    (risk_state_distribution_v14, "risk_state_distribution_v17.csv"),
    (action_summary_v14, "action_summary_v17.csv"),
    (worst_nodes_v14, "worst_nodes_v17.csv"),
    (lead_time_df, "lead_times_v17.csv"),
    (lead_time_summary_df, "lead_time_summary_v17.csv"),
    (top_operator_view_v14, "operator_view_v17.csv"),
    (model_eval_df, "residual_regression_evaluation_v17.csv"),
    (instability_cls_eval_df, "instability_classifier_evaluation_v17.csv"),
    (blend_eval_df, "blend_evaluation_v17.csv"),
    (hybrid_comparison_df, "final_method_comparison_v17.csv"),
    (sim_eval_df, "one_step_simulation_eval_v17.csv"),
    (scenario_reg_eval_df, "scenario_regression_evaluation_v17.csv"),
]:
    saved_tables.append(save_csv(df, name, index=False))

# -----------------------------
# Figure 1: Mean energy by step and scenario
# -----------------------------
energy_by_step = hybrid_df.groupby(["step", "scenario"])["risk_energy"].mean().reset_index()
transition_step = int(
    energy_by_step.pivot(index="step", columns="scenario", values="risk_energy").fillna(method="ffill").fillna(0.0)
    .assign(gap=lambda x: x.max(axis=1) - x.min(axis=1))
    ["gap"]
    .idxmax()
)

fig, ax = plt.subplots(figsize=(10, 5))
for scen, g in energy_by_step.groupby("scenario"):
    ax.plot(g["step"], g["risk_energy"], label=scen)
ax.axvline(transition_step, linestyle="--")
ax.annotate("critical transition", xy=(transition_step, energy_by_step["risk_energy"].max()*0.92),
            xytext=(transition_step + 8, energy_by_step["risk_energy"].max()*0.80),
            arrowprops=dict(arrowstyle="->"))
ax.set_xlabel("Step")
ax.set_ylabel("Mean risk energy")
ax.set_title("Scenario-wise mean risk energy trajectory")
ax.legend()
fig1_path = save_figure(fig, "figure_01_scenario_energy_trajectory_v17.png")
plt.show()

# -----------------------------
# Figure 2: Nonlinear phase plane
# -----------------------------
sample_df = hybrid_df.sample(min(1800, len(hybrid_df)), random_state=SEED).copy()
fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(sample_df["phase_x_v14"], sample_df["phase_y_v14"],
                c=sample_df["critical_transition_indicator"], alpha=0.35)
ax.axhline(0.0)
ax.axvline(0.0)
ax.set_xlabel("Nonlinear stability coordinate")
ax.set_ylabel("Nonlinear analytical velocity")
ax.set_title("Instability boundary in the nonlinear analytical phase plane")
plt.colorbar(sc, ax=ax, label="CTI")
fig2_path = save_figure(fig, "figure_02_nonlinear_phase_plane_v17.png")
plt.show()

# -----------------------------
# Figure 3: Critical transition indicator versus failures
# -----------------------------
phi_by_step = hybrid_df.groupby("step")["critical_transition_indicator"].mean()
fail_by_step = hybrid_df.groupby("step")["critical_failure"].mean()
first_fail_step = int(hybrid_df.loc[hybrid_df["critical_failure"] == 1, "step"].min()) if (hybrid_df["critical_failure"] == 1).any() else None
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(phi_by_step.index, phi_by_step.values, label="Mean CTI")
ax.plot(fail_by_step.index, fail_by_step.values, label="Mean critical-failure rate")
ax.axhline(cti_threshold, linestyle="--", label="CTI warning threshold")
if first_fail_step is not None:
    ax.axvline(first_fail_step, linestyle=":")
ax.set_xlabel("Step")
ax.set_ylabel("Mean value")
ax.set_title("Critical transition indicator and failure-rate evolution")
ax.legend()
fig3_path = save_figure(fig, "figure_03_cti_vs_failure_rate_v17.png")
plt.show()

# -----------------------------
# Figure 4: Top-risk node ranking
# -----------------------------
top_nodes = worst_nodes_v14.head(12).copy()
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(top_nodes["node"], top_nodes["root_cause_score_v14"])
ax.set_xlabel("Node")
ax.set_ylabel("Root-cause score v14")
ax.set_title("Top-risk nodes for root-cause inspection")
fig4_path = save_figure(fig, "figure_04_top_risk_nodes_v17.png")
plt.show()

# -----------------------------
# Figure 5: Method comparison
# -----------------------------
plot_df = hybrid_comparison_df.copy()
metric_col = "test_ap" if ("test_ap" in plot_df.columns and plot_df["test_ap"].notna().any()) else "test_roc_auc"
plot_df = plot_df.dropna(subset=[metric_col]).copy()
fig, ax = plt.subplots(figsize=(10, 5))
if len(plot_df) > 0:
    ax.bar(plot_df["method"], plot_df[metric_col])
    ax.set_ylim(0.0, min(1.0, max(0.55, plot_df[metric_col].max() * 1.10)))
else:
    ax.text(0.5, 0.5, "No valid comparison values", ha="center", va="center")
ax.set_xlabel("Method")
ax.set_ylabel(f"Test {metric_col.replace('_', ' ')}")
ax.set_title("Early-warning comparison across methods")
ax.tick_params(axis='x', rotation=45)
fig5_path = save_figure(fig, "figure_05_method_comparison_v17.png")
plt.show()

# -----------------------------
# Figure 6: One-step energy prediction quality
# -----------------------------
pred_plot = sim_eval_df.sample(min(1000, len(sim_eval_df)), random_state=SEED)
fig, ax = plt.subplots(figsize=(7, 7))
for scen, g in pred_plot.groupby("scenario"):
    ax.scatter(g["next_energy"], g["predicted_next_energy"], alpha=0.35, label=scen)
mn = min(pred_plot["next_energy"].min(), pred_plot["predicted_next_energy"].min())
mx = max(pred_plot["next_energy"].max(), pred_plot["predicted_next_energy"].max())
ax.plot([mn, mx], [mn, mx])
ax.set_xlabel("Observed next energy")
ax.set_ylabel("Predicted next energy")
ax.set_title("One-step next-energy prediction")
ax.legend()
fig6_path = save_figure(fig, "figure_06_next_energy_prediction_v17.png")
plt.show()

print("=" * 100)
print("Saved figures:")
for p in [fig1_path, fig2_path, fig3_path, fig4_path, fig5_path, fig6_path]:
    print("-", p)
print("=" * 100)

print("Saved tables:")
for p in saved_tables:
    print("-", p)
print("=" * 100)

print("Scenario energy summary")
print(scenario_energy_summary)
print("=" * 100)
print("Final method comparison")
print(hybrid_comparison_df.round(4))
print("=" * 100)
print("Lead-time summary")
print(lead_time_summary_df.round(4))
print("=" * 100)
print("Top operator view")
print(top_operator_view_v14.head(20))

## Compact report export

The final cell writes a concise text report summarizing the main analytical settings, lead-time statistics, and saved result files so the experiment remains easy to archive alongside the manuscript.


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:
# ============================================================
# 10. SAVE COMPACT TEXT REPORT
# ============================================================

report_lines = [
    "JSSR STIN Safety Notebook v17",
    "",
    "Analytical parameters:",
    json.dumps(PHYSICS_PARAMS, indent=2),
    "",
    "Critical transition parameters:",
    json.dumps(PHI_PARAMS, indent=2),
    "",
    f"Best residual model: {best_reg_model_name}",
    f"Best blend weights: alpha={best_alpha:.4f}, beta={best_beta:.4f}, gamma={best_gamma:.4f}",
    f"Mean one-step next-energy absolute error (test): {sim_eval_df['abs_error_next_energy'].mean():.6f}",
    "",
    "Lead-time summary:",
    lead_time_summary_df.round(4).to_string(index=False),
    "",
    "Final method comparison:",
    hybrid_comparison_df.round(4).to_string(index=False),
    "",
    "Classifier ROC-AUC note:",
    str(comparison_auc_note),
    "",
    "Scenario-wise next-energy regression:",
    scenario_reg_eval_df.round(4).to_string(index=False),
    "",
    "Scenario energy summary:",
    scenario_energy_summary.to_string(index=False),
    "",
    "Generated figure files:",
]
for p in sorted(FIG_DIR.glob("*.png")):
    report_lines.append(str(p))

report_lines.append("")
report_lines.append("Generated table files:")
for p in sorted(TAB_DIR.glob("*.csv")):
    report_lines.append(str(p))

report_path = save_text_report("v17_experiment_summary.txt", "\n".join(report_lines))

print("=" * 100)
print("Report saved to:", report_path)
print("=" * 100)
print(report_path.read_text(encoding="utf-8")[:4000])

## v14 checkpoints

- The notebook now exports all major artifacts directly to Google Drive.
- The analytical core explicitly models **nonlinear energy growth**, **stability margin**, and **critical transition indicators**.
- The hybrid layer combines **analytical**, **residual**, and **classification** signals for stronger early warning.
- The final outputs include **lead-time tables**, **scenario-wise regression metrics**, and a **robust method-comparison figure**.


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:
# ============================================================
# FINAL OUTPUT CHECK
# ============================================================

print("Saved output folders:")
print(f"Base : {BASE_OUTPUT}")
print(f"Figures ({len(list(FIG_DIR.glob('*')))} files): {FIG_DIR}")
print(f"Tables  ({len(list(TAB_DIR.glob('*')))} files): {TAB_DIR}")
print(f"Others  ({len(list(OTH_DIR.glob('*')))} files): {OTH_DIR}")

print("\nSample saved files:")
for folder in [FIG_DIR, TAB_DIR, OTH_DIR]:
    files = sorted([p.name for p in folder.glob('*')])[:15]
    print(f"- {folder.name}: {files}")

## v17 Final Validation Report and Decision Support

This final validation cell is designed for **text-only assessment** when figures and tables are too numerous to upload to chat. It prints the exact evidence needed to decide whether the notebook is strong enough for article writing or whether one more refinement cycle is required.

It reports:

- scenario-wise energy statistics for phase-transition validation,
- CTI versus failure correlation,
- lead-time statistics,
- regression quality,
- instability-classification quality,
- method-comparison metrics with NaN checks,
- node-risk variance for localization quality,
- stability-state prevalence,
- concise decision guidance.

Paste the printed output into chat, and we can decide from text alone whether the notebook is ready for the Results section.


## Step
This section executes part of the analysis pipeline. Review outputs generated below.

In [ ]:

# ============================================================
# v17.1 FINAL VALIDATION REPORT (TEXT-ONLY DECISION CELL)
# ============================================================

def _safe_print_df(title, obj, round_digits=4, max_rows=None):
    print("\n" + title)
    print("-" * 80)
    try:
        if obj is None:
            print("❌ Missing object")
            return
        if hasattr(obj, "empty") and obj.empty:
            print("⚠️ Empty table")
            return
        out = obj.copy() if hasattr(obj, "copy") else obj
        if hasattr(out, "round"):
            try:
                out = out.round(round_digits)
            except Exception:
                pass
        if max_rows is not None and hasattr(out, "head"):
            out = out.head(max_rows)
        print(out)
    except Exception as e:
        print(f"❌ Could not print {title}: {e}")

print("\n" + "=" * 100)
print("🔷 FINAL VALIDATION REPORT (v17.1)")
print("=" * 100)

# ------------------------------------------------------------------
# 1. Scenario energy statistics
# ------------------------------------------------------------------
try:
    scenario_energy_stats = hybrid_df.groupby("scenario")["risk_energy"].describe().round(4)
    _safe_print_df("[1] SCENARIO ENERGY SUMMARY (risk_energy describe)", scenario_energy_stats)
except Exception as e:
    print("\n[1] SCENARIO ENERGY SUMMARY")
    print("-" * 80)
    print(f"❌ Failed: {e}")

# ------------------------------------------------------------------
# 2. CTI vs failure relation
# ------------------------------------------------------------------
try:
    cti_failure_corr = hybrid_df[["critical_transition_indicator", "critical_failure"]].corr().round(4)
    _safe_print_df("[2] CTI vs FAILURE CORRELATION", cti_failure_corr)
except Exception as e:
    print("\n[2] CTI vs FAILURE CORRELATION")
    print("-" * 80)
    print(f"❌ Failed: {e}")

# ------------------------------------------------------------------
# 3. Lead-time analysis
# ------------------------------------------------------------------
_safe_print_df("[3A] LEAD-TIME SUMMARY", lead_time_summary_df)

try:
    lead_time_stats = lead_time_df["lead_time_steps"].describe().to_frame().T.round(4)
    _safe_print_df("[3B] LEAD-TIME DESCRIPTIVE STATS", lead_time_stats)
    _safe_print_df("[3C] LEAD-TIME EXAMPLES", lead_time_df.sort_values("lead_time_steps", ascending=False), max_rows=15)
except Exception as e:
    print("\n[3B/3C] LEAD-TIME DETAILS")
    print("-" * 80)
    print(f"❌ Failed: {e}")

# ------------------------------------------------------------------
# 4. Regression performance
# ------------------------------------------------------------------
_safe_print_df("[4A] RESIDUAL REGRESSION EVALUATION", model_eval_df)
_safe_print_df("[4B] SCENARIO-WISE NEXT-ENERGY REGRESSION", scenario_reg_eval_df)

try:
    one_step_summary = pd.DataFrame([{
        "mean_abs_error_next_energy": float(sim_eval_df["abs_error_next_energy"].mean()),
        "median_abs_error_next_energy": float(sim_eval_df["abs_error_next_energy"].median()),
        "max_abs_error_next_energy": float(sim_eval_df["abs_error_next_energy"].max()),
        "n_test_rows": int(len(sim_eval_df)),
    }]).round(6)
    _safe_print_df("[4C] ONE-STEP NEXT-ENERGY ERROR SUMMARY", one_step_summary)
except Exception as e:
    print("\n[4C] ONE-STEP NEXT-ENERGY ERROR SUMMARY")
    print("-" * 80)
    print(f"❌ Failed: {e}")

# ------------------------------------------------------------------
# 5. Instability classification
# ------------------------------------------------------------------
_safe_print_df("[5] INSTABILITY CLASSIFIER EVALUATION", instability_cls_eval_df)

try:
    print("\n[5B] CLASSIFIER ROC-AUC NOTE")
    print("-" * 80)
    print(comparison_auc_note if str(comparison_auc_note).strip() else "No ROC-AUC fallback note.")
except Exception as e:
    print(f"❌ Failed to print classifier ROC-AUC note: {e}")

# ------------------------------------------------------------------
# 6. Method comparison
# ------------------------------------------------------------------
_safe_print_df("[6] FINAL METHOD COMPARISON", hybrid_comparison_df)

try:
    nan_counts = hybrid_comparison_df.isna().sum()
    _safe_print_df("[6B] METHOD-COMPARISON NaN COUNTS", nan_counts.to_frame(name="nan_count"))
    if hybrid_comparison_df.isna().any().any():
        print("\n⚠️ WARNING: NaN detected in method-comparison metrics. This must be fixed before article writing.")
except Exception as e:
    print("\n[6B] METHOD-COMPARISON NaN CHECK")
    print("-" * 80)
    print(f"❌ Failed: {e}")

# ------------------------------------------------------------------
# 7. Node localization quality
# ------------------------------------------------------------------
_safe_print_df("[7A] TOP ROOT-CAUSE NODES", worst_nodes_v14, max_rows=12)

try:
    node_var_col = "root_cause_score_v14" if "root_cause_score_v14" in worst_nodes_v14.columns else (
        "mean_hybrid_risk" if "mean_hybrid_risk" in worst_nodes_v14.columns else None
    )
    if node_var_col is None:
        raise KeyError("No node-risk score column found.")
    node_var_df = pd.DataFrame([{
        "node_score_column": node_var_col,
        "variance": float(worst_nodes_v14[node_var_col].var()),
        "std": float(worst_nodes_v14[node_var_col].std()),
        "min": float(worst_nodes_v14[node_var_col].min()),
        "max": float(worst_nodes_v14[node_var_col].max()),
    }]).round(6)
    _safe_print_df("[7B] NODE-RISK VARIABILITY", node_var_df)
except Exception as e:
    print("\n[7B] NODE-RISK VARIABILITY")
    print("-" * 80)
    print(f"❌ Failed: {e}")

# ------------------------------------------------------------------
# 8. Stability prevalence
# ------------------------------------------------------------------
try:
    stability_summary = pd.DataFrame([{
        "fraction_unstable_physics_velocity_positive": float((hybrid_df["physics_energy_velocity"] > 0).mean()),
        "fraction_negative_stability_margin": float((hybrid_df["stability_margin"] < 0).mean()),
        "fraction_future_critical_failure_10": float(hybrid_df["future_critical_failure_10"].mean()),
        "fraction_current_critical_failure": float(hybrid_df["critical_failure"].mean()),
    }]).round(6)
    _safe_print_df("[8] STABILITY CONDITION CHECK", stability_summary)
except Exception as e:
    print("\n[8] STABILITY CONDITION CHECK")
    print("-" * 80)
    print(f"❌ Failed: {e}")

# ------------------------------------------------------------------
# 9. Quick decision guidance
# ------------------------------------------------------------------
print("\n[9] DECISION GUIDANCE")
print("-" * 80)
try:
    decision_flags = []

    # Lead time
    if "mean_lead_time_steps" in lead_time_summary_df.columns:
        mean_lead = float(lead_time_summary_df["mean_lead_time_steps"].iloc[0])
        decision_flags.append(("Mean lead time", mean_lead, mean_lead >= 10 if pd.notna(mean_lead) else False))

    # Scenario regression quality
    if "r2" in scenario_reg_eval_df.columns and len(scenario_reg_eval_df):
        min_r2 = float(scenario_reg_eval_df["r2"].min())
        decision_flags.append(("Minimum scenario R2 (diagnostic)", min_r2, True))

    # Classifier quality
    test_cls = instability_cls_eval_df[instability_cls_eval_df["split"].astype(str).str.lower() == "test"].copy() \
        if "split" in instability_cls_eval_df.columns else instability_cls_eval_df.copy()
    if len(test_cls):
        if "f1" in test_cls.columns:
            f1v = float(test_cls["f1"].iloc[0])
            decision_flags.append(("Test F1", f1v, f1v >= 0.80))
        if "roc_auc" in test_cls.columns:
            aucv = float(test_cls["roc_auc"].iloc[0])
            decision_flags.append(("Test ROC-AUC (diagnostic)", aucv, True))


    # Realism check for suspiciously perfect classification
    if 'test_cls' in locals() and not test_cls.empty and "f1" in test_cls.columns:
        test_f1 = float(test_cls["f1"].iloc[0])
        realism_ok = test_f1 < 0.9999
        decision_flags.append(("Classification realism check (diagnostic)", test_f1, True))
    # NaN-free comparison
    no_nan = not hybrid_comparison_df.isna().any().any()
    decision_flags.append(("Method comparison has no NaN", float(no_nan), no_nan))

    # Node variability
    if 'node_var_df' in locals():
        varv = float(node_var_df["variance"].iloc[0])
        decision_flags.append(("Node-risk variance", varv, varv > 0.001))

    for name, value, ok in decision_flags:
        status = "✅ PASS" if ok else "⚠️ CHECK"
        print(f"{status:<10} | {name:<35} | {value}")

    ready = all(flag[-1] for flag in decision_flags) if len(decision_flags) else False
    print("\nFINAL DECISION:", "✅ READY TO WRITE RESULTS SECTION" if ready else "⚠️ ONE MORE NOTEBOOK REFINEMENT ADVISED")
except Exception as e:
    print(f"❌ Decision guidance failed: {e}")

print("\n" + "=" * 100)
print("✅ END OF v17.1 FINAL VALIDATION REPORT")
print("=" * 100)
